# Name That LLM 基础 EDA

S03 初版统计，不训练模型、不修改原始数据。所有带标签结论仅来自训练集；3 条示例测试只检查结构。词数按空白切分，字符数按 Python 字符计数，ASCII 标点只计 `.,;:!?`。统计关联不代表可识别性或泛化收益。

本 Notebook 复用基线的路径和输入函数，借助检查脚本装载函数定义，S06 前不另建公共业务包。

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
from ntl.data import resolve_data_dir, load_competition_data, file_fingerprint
from ntl.analysis import basic_analysis

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'pyproject.toml').is_file())
DATA_DIR = resolve_data_dir(os.environ.get('NAME_THAT_LLM_DATA_DIR'))
train, test, sample, input_summary = load_competition_data(DATA_DIR)
original_hashes = {name: item['sha256'] for name, item in input_summary['files'].items()}
report, lengths, by_model = basic_analysis(train, test)
OUTPUT = Path(os.environ.get('NAME_THAT_LLM_EDA_OUTPUT_DIR', str(ROOT / 'artifacts/s06b/eda')))
OUTPUT.mkdir(parents=True, exist_ok=True)
print('训练样本:', len(train), '类别:', train['model'].nunique(), 'prompt:', train['prompt_id'].nunique())
print('示例测试仅用于结构检查:', len(test), test.columns.tolist())


## 类别及 prompt 分布

In [ ]:
display(train['model'].value_counts().sort_index().rename('samples').to_frame())
print('每 prompt 样本数分布:', report['prompt_size_distribution'])
print('精确完整性检查:', json.dumps(report['issues'], ensure_ascii=False))

## 长度的总体分布

In [ ]:
display(lengths[['characters', 'words', 'newlines', 'ascii_punctuation']].describe())
for column, bins in [('words', [0, 50, 100, 150, 200, 250, float('inf')]),
                     ('characters', [0, 400, 800, 1200, 1600, float('inf')])]:
    display(pd.cut(lengths[column].astype(float), bins=bins).value_counts(sort=False).rename('samples').to_frame())

## 模型间长度及低成本格式统计

In [ ]:
display(by_model)
lengths.to_csv(OUTPUT / 'text_statistics.csv', index=False)
by_model.to_csv(OUTPUT / 'model_length_statistics.csv')
(OUTPUT / 'basic-summary.json').write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')

## 观察 假设和研究边界

In [ ]:
means = lengths.groupby('model')['words'].mean().sort_values()
print(f'平均词数最低: {means.index[0]} {means.iloc[0]:.2f}; 最高: {means.index[-1]} {means.iloc[-1]:.2f}')
print('全部训练文本换行总数:', int(lengths['newlines'].sum()))
print('假设：长度和标点分布可能存在模型风格关联；需 S07 在固定分折中做消融，当前不代表分类能力。')
print('若换行数恒为零，则当前数据上的换行计数不提供样本间差异。')
print('延后：语义近重复、词性/句法、深入 Markdown 及复杂标点研究；不影响初版分折验收。')
for name, expected in original_hashes.items():
    assert file_fingerprint(DATA_DIR / name)['sha256'] == expected
print('原始文件 SHA-256 未变；EDA 完成。')